# Performance profile

The ISMIP7 forcing datasets on [source.coop](https://source.coop/ismip) are about
285,000 NetCDF files (6 TB). The [englacial/ismip7](https://source.coop/englacial/ismip7)
Icechunk stores hold *virtual references* into those files (22 GB of metadata, no data
copied). One store per ice sheet, `icechunk-ais` and `icechunk-gris`, laid out as

    <ESM>/<experiment>/<product>/<resolution>m      e.g. CESM2-WACCM/ssp585/SDBN1/2000m

Each resolution group is one xarray Dataset: a normalised monthly (or annual) time axis
in `days since 1850-01-01`, the exact ISMIP7 grid in `x`/`y`, and a `crs` variable with
the EPSG code and WKT. The group above the resolutions (the *family*, e.g.
`CESM2-WACCM/ssp585/SDBN1`) carries an OME-NGFF `multiscales` attribute naming its
levels, so a multiscale-aware reader (gridlook) opens the family as one pyramid.

Reading needs no credentials: `icechunk` opens the store, `xarray` reads it. The
hierarchy is walked with [rustytree](https://github.com/aladinor/rustytree) when it is
installed (a concurrent walk in Rust: the whole store in seconds), falling back to
zarr-python, which does the same one group at a time.


In [ ]:
import os
import time

import icechunk
import xarray as xr

REGION = os.environ.get("ISMIP7_NB_REGION", "AIS")  # or "GrIS"
BUCKET = "us-west-2.opendata.source.coop"
PREFIX = f"englacial/ismip7/icechunk-{REGION.lower()}"
# the bucket the virtual chunks point into; icechunk only follows references into
# containers you authorise explicitly
SOURCE = "s3://us-west-2.opendata.source.coop/ismip/"


# the series run to 2300, past datetime64[ns]: decode time with cftime
TIME = xr.coders.CFDatetimeCoder(use_cftime=True)


def read_time(a, tries=3):
    """Seconds to load an xarray object into memory (retrying transient S3 errors)."""
    for attempt in range(tries):
        t0 = time.time()
        try:
            a.load()
            return time.time() - t0
        except Exception:
            if attempt == tries - 1:
                raise
            time.sleep(2)


storage = icechunk.s3_storage(bucket=BUCKET, prefix=PREFIX, region="us-west-2", anonymous=True)
containers = icechunk.containers_credentials({SOURCE: icechunk.s3_anonymous_credentials()})
repo = icechunk.Repository.open(storage, authorize_virtual_chunk_access=containers)
session = repo.readonly_session("main")
print("store:", f"s3://{BUCKET}/{PREFIX}", "| snapshot", session.snapshot_id)

In [ ]:
import zarr

root = zarr.open_group(session.store, mode="r")

families, levels = {}, []
for path, node in root.members(max_depth=None):
    if not isinstance(node, zarr.Group):
        continue
    if "multiscales" in node.attrs:
        families[path] = [d["path"] for d in node.attrs["multiscales"][0]["datasets"]]
    elif node.attrs.get("ismip7_group"):
        levels.append(path)
print(len(levels), "data groups,", len(families), "families")
for fam, lv in list(families.items())[:8]:
    print(f"  {fam}: {lv}")
FAMILY = os.environ.get("ISMIP7_NB_FAMILY", "CESM2-WACCM/historical/SDBN1")
GROUP = f"{FAMILY}/{families[FAMILY][-1]}"  # the coarsest level
print("using", GROUP)

What a reader pays: opening the repository, opening one group, the first time step of the first 3-D variable, and a year of a 200 x 200 window; with the zarr engine and, when it is installed, [rustytree](https://github.com/aladinor/rustytree), whose arrays read through its own Rust path (zarrs + icechunk) rather than Python zarr. Then how big the manifests are (what a browser client downloads before the first step).

In [ ]:
import pandas as pd

# the "year" read is a 200 x 200 cell window so the notebook stays small enough for
# Binder (a full year of a 2 km field is 444 MB, of the 3-D ocean fields 834 MB)
ENGINES = ["zarr"]
try:
    import rustytree  # noqa: F401

    ENGINES.append("rustytree")
except ImportError:
    pass

results = []
for gp in levels[:10]:
    for engine in ENGINES:
        t0 = time.time()
        r = icechunk.Repository.open(storage, authorize_virtual_chunk_access=containers)
        sess = r.readonly_session("main")
        t_open = round(time.time() - t0, 2)
        t0 = time.time()
        if engine == "zarr":
            d = xr.open_zarr(sess.store, group=gp, consolidated=False, decode_times=TIME)
        else:
            d = xr.open_dataset(sess, group=gp, engine=engine, decode_times=TIME)
        t_group = round(time.time() - t0, 2)
        v = next(x for x in d.data_vars if d[x].ndim >= 3)
        tdim = d[v].dims[0]
        t_first = read_time(d[v].isel({tdim: 0}))
        n = min(12, d.sizes[tdim])
        window = {tdim: slice(0, n), "x": slice(100, 300), "y": slice(100, 300)}
        t_year = read_time(d[v].isel(window))
        results.append(
            {
                "group": gp,
                "engine": engine,
                "var": v,
                "steps": d.sizes[tdim],
                "chunks": d[v].encoding.get("chunks"),
                "open": t_open,
                "group_open": t_group,
                "first_step": round(t_first, 2),
                "year_window": round(t_year, 2),
            }
        )
pd.DataFrame(results)

## Manifest sizes

Icechunk splits each array's manifest every 120 chunks along time; a client reads one split per time step it touches.

In [ ]:
sizes = sorted((m.size_bytes for m in repo.list_manifest_files(session.snapshot_id)), reverse=True)
print(
    f"{len(sizes)} manifests; total {sum(sizes) / 1e6:.1f} MB; "
    f"largest {sizes[0] / 1e6:.2f} MB; median {sizes[len(sizes) // 2] / 1e3:.0f} kB"
)